# First order difference

In [1]:
from util.data_fft import *
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft, fftfreq
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import RidgeClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import itertools
from itertools import chain, combinations
from sklearn.preprocessing import normalize
import pickle
import copy
import pandas as pd
from IPython.display import display, HTML
from scipy.signal import find_peaks
from scipy.stats import skew, entropy, kurtosis
import seaborn as sns
import pyxai
from pyxai import Learning, Explainer, Tools
from util.explain_models import *
import tqdm
import shap
import time
from sklearn.tree import export_graphviz
import graphviz

/Users/morvan/Documents/Recherche/Publications/XAI_ITSC_2026/itsc-xai-generators/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load raw data

In [51]:
data_fft = pd.read_csv('data/itsc_generators_edf_v1.0.csv')
display(data_fft)

,3.3 Hz,6.7 Hz,10 Hz,13.3 Hz,16.7 Hz,20 Hz,23.3 Hz,26.7 Hz,30.0 Hz,33.3 Hz,36.7 Hz,40 Hz,43.3 Hz,46.7 Hz,y,Machine
0,0.003823,0.006244,0.009360,0.006350,0.011895,0.003084,0.004992,0.003583,0.002899,0.004486,0.002296,0.003973,0.002023,0.009182,0,E
1,0.003120,0.004690,0.006866,0.004514,0.008899,0.001253,0.003436,0.002102,0.002641,0.003109,0.000343,0.002801,0.005365,0.002798,0,E
2,0.003904,0.006731,0.009307,0.006487,0.010539,0.002833,0.005094,0.003233,0.002605,0.003909,0.002039,0.003685,0.001897,0.007596,0,E
3,0.002495,0.004428,0.006738,0.005906,0.009899,0.002494,0.005030,0.003931,0.000839,0.004511,0.003686,0.004014,0.001420,0.012222,0,E
4,0.002107,0.003078,0.003943,0.003887,0.008632,0.002485,0.001238,0.002171,0.000282,0.002441,0.000674,0.003722,0.003891,0.011018,0,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,0.009815,0.010288,0.004735,0.004409,0.018036,0.022986,0.019295,0.004795,0.005513,0.009084,0.012807,0.009289,0.001562,0.011203,1,A
86,0.003257,0.001131,0.001011,0.000957,0.005961,0.010337,0.008621,0.002342,0.005213,0.003704,0.004338,0.002292,0.001172,0.003184,1,A
87,0.005867,0.003401,0.001492,0.003290,0.012677,0.017022,0.014693,0.003215,0.006826,0.005421,0.009559,0.007115,0.001118,0.009298,1,A
88,0.002854,0.000671,0.001469,0.000268,0.005062,0.009380,0.008365,0.003199,0.002830,0.002925,0.001969,0.003003,0.003951,0.006794,1,A


## Feature selection

In [39]:
data_diff = data_fft.drop(data_fft.columns[-1:], axis=1)
data_diff = first_order_difference(data_diff)
data_diff = data_diff.drop(data_diff.columns[-1:], axis=1)
display(data_diff)
print("Generating power sets")
data_diff_ps = column_power_set(data_diff, 5, 5)
n_sets = len(data_diff_ps)
print("number of feature sets", n_sets)

,6.7 Hz,10 Hz,13.3 Hz,16.7 Hz,20 Hz,23.3 Hz,26.7 Hz,30.0 Hz,33.3 Hz,36.7 Hz,40 Hz,43.3 Hz,46.7 Hz
0,0.002422,0.003116,-0.003010,0.005545,-0.008810,0.001908,-0.001409,-0.000683,0.001587,-0.002190,0.001677,-0.001950,0.007159
1,0.001571,0.002175,-0.002351,0.004384,-0.007646,0.002183,-0.001334,0.000539,0.000468,-0.002766,0.002457,0.002564,-0.002566
2,0.002827,0.002576,-0.002820,0.004052,-0.007706,0.002261,-0.001861,-0.000628,0.001303,-0.001870,0.001646,-0.001788,0.005699
3,0.001932,0.002311,-0.000833,0.003993,-0.007405,0.002536,-0.001099,-0.003092,0.003672,-0.000825,0.000329,-0.002594,0.010802
4,0.000971,0.000864,-0.000055,0.004745,-0.006147,-0.001248,0.000934,-0.001889,0.002159,-0.001767,0.003048,0.000169,0.007127
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,0.000473,-0.005553,-0.000325,0.013627,0.004950,-0.003692,-0.014500,0.000719,0.003571,0.003723,-0.003517,-0.007727,0.009640
86,-0.002126,-0.000120,-0.000054,0.005005,0.004376,-0.001716,-0.006279,0.002871,-0.001509,0.000633,-0.002046,-0.001120,0.002012
87,-0.002465,-0.001909,0.001798,0.009386,0.004345,-0.002329,-0.011477,0.003611,-0.001405,0.004138,-0.002444,-0.005997,0.008180
88,-0.002183,0.000798,-0.001201,0.004794,0.004318,-0.001015,-0.005165,-0.000370,0.000095,-0.000956,0.001034,0.000948,0.002842


Generating power sets
number of feature sets 1287


In [44]:
best_features_min = {'features':None, 'min_accuray':0, 'mean_accuracy':0}
for d in data_diff_ps:
    data = pd.concat([d, data_fft['y'], data_fft['Machine']], axis=1)

    min_a, mean_a = cross_validation(data, n_runs=1, data_representation='raw');
    if min_a >= best_features_min['min_accuray']:
        best_features_min['features'] = d.columns.tolist()
        best_features_min['min_accuray'] = min_a
        best_features_min['mean_accuracy'] = mean_a
print(best_features_min)

{'features': ['13.3 Hz', '26.7 Hz', '30.0 Hz', '40 Hz', '46.7 Hz'], 'min_accuray': 0.26666666666666666, 'mean_accuracy': 0.6666666666666666}


The selection procedure fails to converge to a feature set that generalizes well